# Task 1 entry point — preserved Step 3 dataset
Reorganized only; no experiment cells executed during creation. Existing results were copied with byte-hash verification.

In [ ]:
from pathlib import Path
import json, hashlib, random, gc, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D
from IPython.display import display
from tqdm.auto import tqdm
import open_clip

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if p.name == 'Task 1'
             and (p / 'task1/configs/protocol.json').is_file()), None)
if REPO is None:
    raise RuntimeError('Start Jupyter inside Task 1.')
TASK_DIR = REPO / 'task1'
DATA_DIR = TASK_DIR / 'data'  # Existing official dataset; read-only.
BASELINE_PATH = TASK_DIR / 'results/clean_baseline.pt'
PRESERVED = TASK_DIR / 'data/preserved_cue_conflicts'
SEED = 6304
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = EXTRACTION_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 256
MAX_EPOCHS, PATIENCE = 50, 5
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
HUE_FACTOR = 0.25
GRID_SIZE, PATCH_SIZE = 4, 56
DISPLACEMENTS = (0, 8, 16, 32)
DIRECTIONS = {'left':(-1,0), 'right':(1,0), 'up':(0,-1), 'down':(0,1)}
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def load_notebook(relative):
    path = TASK_DIR / relative
    for index, cell in enumerate(json.loads(path.read_text(encoding='utf-8'))['cells']):
        if cell['cell_type'] == 'code':
            code = cell['source']
            exec(compile(''.join(code) if isinstance(code,list) else code,
                         f'{path}:cell-{index}', 'exec'), globals())

for helper in ['data/make_subset.ipynb', 'data/make_cue_conflicts.ipynb',
               'data/transforms.ipynb', 'models/backbones.ipynb',
               'analysis/evaluate_bias.ipynb', 'analysis/feature_similarity.ipynb',
               'analysis/representation.ipynb']:
    load_notebook(helper)
seed_everything()


In [ ]:
# Restart kernel between steps; run all cells with the chosen STEP.
# clean -> color -> cue_conflicts -> translation -> patch_shuffle -> representation
STEP = 'clean'
load_fixed_subset()
# Always verify the protected Step 3 archive before proceeding.
load_preserved_conflicts()
actions = {'clean':run_clean, 'color':run_color, 'cue_conflicts':run_cue_conflicts,
           'translation':run_translation, 'patch_shuffle':run_patch_shuffle,
           'representation':run_representation}
actions[STEP]()
